# OLMo: reference frames, persistent homology, RMT and NTK

Keep `structure_study.py` and `forgetting_mechanisms.py` beside this notebook. This is an **observational checkpoint comparison** inspired by the attention-sink paper. Fresh random models are controls, not the original pretraining initialization.

Launch returns immediately. Rerun Refresh status manually. No continuous output stream. Read `README.md` for the precise geometry, topology, phase and random-matrix definitions.

In [ ]:
from pathlib import Path
import structure_study as study
import importlib
importlib.reload(study)

# Locate the ORIGINAL association campaign, not a measurement/results ZIP.
search_roots = [Path.cwd(), Path.cwd().parent, Path.home() / "1"]
possible = [p / "runs" / "olmo_association_v1" for p in search_roots]
possible += [Path("/home/ubuntu/1/runs/olmo_association_v1")]
found = sorted({p.resolve() for p in possible if (p / "config.json").exists()})
print("Available source campaigns:", *found, sep="\n")
SOURCE = found[0] if found else None
# If not found, set SOURCE = Path("/your/actual/original/association/campaign")


## Configure

Use the discovered source or explicitly set its actual path. The default compares two random initializations, pretrained, seed 1 A-anchor, and states after B updates 21 and 47. Extend to all three seeds after timing a first run.

In [ ]:
if SOURCE is None or not (SOURCE / "config.json").exists():
    raise FileNotFoundError("Set SOURCE above to your original campaign containing config.json, seed1/anchor.pt and saved forks.")
previous_settings = globals().get("SETTINGS", {})
OUTPUT = (Path(previous_settings["output"])
          if previous_settings.get("source") == str(SOURCE)
          else SOURCE.parent / "olmo_structure_v1")
SETTINGS = study.default_settings(SOURCE, OUTPUT)
SETTINGS.update(
    seeds=[1],                       # Expand to [1, 2, 3] if those checkpoints exist
    events=[21, 47],                 # States after these actual updates
    device="cuda:0",
    threads=8,
    top_k=16,                        # Full-matrix randomized leading SVD
    matrix_sample=128,               # Exact covariance spectrum on fixed subsamples
    topology_points=16,              # 4..32; larger costs much more
    prompts_per_domain=8,            # Fixed A/B geometry panel; enough prompts for held-out alignment
    activation_positions=8,
)
print("Source:", SOURCE)
print("Output:", OUTPUT)
print("Jobs:", len(study.plan(SETTINGS)[0]) + len(study.plan(SETTINGS)[1]))
# Additional analyses, integrated into the same background run.
SETTINGS["extended"].update(
    enabled=True,                   # Joint PH/RMT and held-out frame alignment
    null_repeats=8,                  # Spectrum-preserving, shuffled, Gaussian controls
    ntk_enabled=True,
    ntk_mode="auto",                 # Exact only for tiny models; sketches for OLMo
    ntk_prompts_per_domain=4,         # Fixed A and B prompts, two outputs each
    ntk_sketch_dim=512,              # Increase to 2048+ if projection audit fails
)


## Optional: real intermediate pretraining / A-continuation checkpoints

Disabled by default. Use actual local paths. Fresh random initialization plus a final pretrained checkpoint does not establish when structure emerged.

In [ ]:
ADD_CHECKPOINTS = False
if ADD_CHECKPOINTS:
    SETTINGS["extra_checkpoints"] = [
        dict(id="pretrain_step10000", kind="hf",
             path="/actual/local/HF_checkpoint_directory", compare_to="pretrained"),
        dict(id="seed1_A_continuation", kind="checkpoint", seed=1,
             path="/actual/local/A_continuation_checkpoint.pt", compare_to="seed1_A_anchor"),
    ]

# Optional: replace the tiny association geometry panel with your fixed texts.
# Every text must fit the source max_length; nothing is silently truncated.
# Full original A/B test loss is still measured separately.
# SETTINGS["geometry_texts"] = ["Your first fixed passage...", "Your second fixed passage..."]

## Run / restart the experiment

Run the next cell to start the study. Run it again to stop the current study and start fresh. Previous results are preserved in their existing folder. The cell returns after the background study starts.


In [ ]:
OUTPUT = study.restart(SETTINGS)


## Refresh status — rerun manually

Watch phase/matrix/prompt. A fresh heartbeat shows liveness, not necessarily progress.

In [ ]:
study.print_status(SETTINGS["output"])

## Show results — rerun after jobs finish

Compact loss tables plus plots by default. Set `FULL=True` for the matrix tables. The JSON/CSV files always contain all measurements.
Includes PH/RMT frame comparisons and NTK summaries/plots. NTK audit=False means projection uncertainty is too large for small-effect conclusions.


In [ ]:
FULL = False
study.show(SETTINGS["output"], full=FULL)

## Optional diagnostics and stop

Set STOP=True to stop the experiment without restarting. Set SHOW_LOG=True to inspect recent logs. Both are disabled by default.


In [ ]:
OUTPUT = Path(SETTINGS["output"])
SHOW_LOG = False
STOP = False
if SHOW_LOG:
    state = study.status(OUTPUT)
    path = OUTPUT / state["job"] / "worker.log" if state.get("job") else OUTPUT / "launcher.log"
    print("\n".join(path.read_text().splitlines()[-40:]) if path.exists() else "No log yet")
if STOP:
    print(study.stop(OUTPUT))
# Download all numbers, metadata and plots as one archive, without rerunning.
EXPORT = False
if EXPORT:
    import shutil
    archive = shutil.make_archive(str(OUTPUT) + "_results", "zip", OUTPUT)
    print("Results archive:", archive)


## Optional implementation checks

These check finite topology, invariance, covariance normalization and tied weights. They do not test scientific hypotheses. For a tiny-model end-to-end test, run in a terminal:

`python structure_study.py smoke --output /path/to/a/new/tiny_test`

In [ ]:
RUN_TESTS = False
if RUN_TESTS:
    study.self_test()